In [25]:
from statistics import mean, pstdev, stdev
import re
import numpy as np
import pandas as pd
from scipy import stats

In [26]:
# One-sided Welch's t-test for improvement (lower is better).

alpha = 0.05

def welch_ttest_less(x_old, x_new):
    """
    One-sided Welch's t-test for improvement (lower is better).
    Tests H1: mean(new) < mean(old).
    Returns a dict of summary stats.
    """
    res = stats.ttest_ind(x_new, x_old, equal_var=False, alternative='less')
    n_old, n_new = len(x_old), len(x_new)
    m_old, m_new = np.mean(x_old), np.mean(x_new)
    v_old, v_new = np.var(x_old, ddof=1), np.var(x_new, ddof=1)

    # Welch–Satterthwaite df
    df = (v_old/n_old + v_new/n_new)**2 / (
        (v_old**2)/((n_old**2)*(n_old-1)) + (v_new**2)/((n_new**2)*(n_new-1))
    )

    return {
        "mean_old": m_old,
        "mean_new": m_new,
        "diff_new_minus_old": m_new - m_old,  # negative => faster
        "t_stat": res.statistic,
        "df": float(df),
        "p_value_one_sided": res.pvalue,
        "Conclusion (α=0.05)": "Reject H0" if res.pvalue < alpha else "Fail to reject H0"
    }

# Experiment 1: Double vs Triple (lower = faster)

In [27]:
double_time = np.array([24.68,24.68,24.92,25.09,25.38,25.45,25.45,25.47,24.63,24.49,24.54,24.65,24.97,25.10,25.25,25.20,25.03,25.23,25.36,25.23,25.35,25.51,25.65,25.61])
double_compute = np.array([16.905,17.052,17.125,17.220,17.388,17.449,17.556,17.498,17.044,17.091,17.149,17.197,17.398,17.526,17.574,17.520,17.068,17.131,17.181,17.274,17.427,17.554,17.586,17.609])

triple_time = np.array([24.26,24.29,24.40,24.68,24.98,24.93,25.28,25.00,24.64,24.44,24.69,24.61,24.98,25.06,24.85,24.99,24.37,24.49,24.54,24.59,24.82,24.84,25.42,25.09])
triple_compute = np.array([16.727,16.853,16.953,17.052,17.322,17.388,17.376,17.410,16.961,17.073,17.159,17.126,17.445,17.494,17.426,17.456,16.902,16.986,17.103,17.151,17.286,17.339,17.481,17.451])

exp1_time = welch_ttest_less(double_time, triple_time)
exp1_compute = welch_ttest_less(double_compute, triple_compute)

exp1_df = pd.DataFrame([
    {"Metric": "Time per Image", **exp1_time},
    {"Metric": "Compute", **exp1_compute},
])

exp1_df

,Metric,mean_old,mean_new,diff_new_minus_old,t_stat,df,p_value_one_sided,Conclusion (α=0.05)
0,Time per Image,25.121667,24.760,-0.361667,-3.760824,45.198650,0.000242,Reject H0
1,Compute,17.313417,17.205,-0.108417,-1.687982,45.818106,0.049106,Reject H0


# Experiment 2: Memory Handling Logic (Old vs New)

In [28]:
old = np.array([29.27,29.5,28.2,28.91,29.06,28.91,28.61,29.27,28.0,28.8,28.19,28.7,28.57,27.6,28.76,29.85,23.36,25.01,25.08,26.14,26.21,26.21,25.29,25.4])
new = np.array([22.92,23.65,25.18,25.6,26.28,26.81,26.6,22.14,23.3,23.48,25.45,25.57,25.46,23.76,22.74,23.1,24.02,22.74,23.1,24.02,25.47,26.04,26.3,26.17])

exp2 = welch_ttest_less(old, new)
exp2_df = pd.DataFrame([{"Metric": "Time per Image (Experiment 2)", **exp2}])

print("\nExperiment 2 — Memory Handling (H1: newest < old; α=0.05)")
exp2_df


Experiment 2 — Memory Handling (H1: newest < old; α=0.05)


,Metric,mean_old,mean_new,diff_new_minus_old,t_stat,df,p_value_one_sided,Conclusion (α=0.05)
0,Time per Image (Experiment 2),27.620833,24.579167,-3.041667,-6.444235,44.216656,3.670667e-08,Reject H0


# Processing the Data of the final Run

In [29]:
raw_final = """
Name	Duration
PROCESS_FILE:20170906_12_01_00.npz	25.997 s
GPU_COMPUTE	17.536 s
PROCESS_FILE:20170906_12_01_24.npz	27.880 s
GPU_COMPUTE	18.232 s
PROCESS_FILE:20170906_12_01_48.npz	28.855 s
GPU_COMPUTE	18.583 s
PROCESS_FILE:20170906_12_02_12.npz	29.825 s
GPU_COMPUTE	18.957 s
PROCESS_FILE:20170906_12_02_36.npz	30.729 s
GPU_COMPUTE	19.696 s
PROCESS_FILE:20170906_12_03_00.npz	31.951 s
GPU_COMPUTE	19.916 s
PROCESS_FILE:20170906_12_03_24.npz	28.498 s
GPU_COMPUTE	18.247 s
PROCESS_FILE:20170906_12_03_48.npz	29.007 s
GPU_COMPUTE	18.518 s
PROCESS_FILE:20170906_12_01_00.npz	25.953 s
GPU_COMPUTE	17.895 s
PROCESS_FILE:20170906_12_01_24.npz	27.751 s
GPU_COMPUTE	18.376 s
PROCESS_FILE:20170906_12_01_48.npz	28.690 s
GPU_COMPUTE	18.710 s
PROCESS_FILE:20170906_12_02_12.npz	29.427 s
GPU_COMPUTE	19.030 s
PROCESS_FILE:20170906_12_02_36.npz	30.654 s
GPU_COMPUTE	19.397 s
PROCESS_FILE:20170906_12_03_00.npz	31.753 s
GPU_COMPUTE	19.778 s
PROCESS_FILE:20170906_12_03_24.npz	28.694 s
GPU_COMPUTE	18.206 s
PROCESS_FILE:20170906_12_03_48.npz	28.853 s
GPU_COMPUTE	18.698 s
PROCESS_FILE:20170906_12_01_00.npz	25.717 s
GPU_COMPUTE	17.528 s
PROCESS_FILE:20170906_12_01_24.npz	27.694 s
GPU_COMPUTE	18.237 s
PROCESS_FILE:20170906_12_01_48.npz	28.621 s
GPU_COMPUTE	18.599 s
PROCESS_FILE:20170906_12_02_12.npz	29.550 s
GPU_COMPUTE	18.846 s
PROCESS_FILE:20170906_12_02_36.npz	30.678 s
GPU_COMPUTE	19.355 s
PROCESS_FILE:20170906_12_03_00.npz	31.917 s
GPU_COMPUTE	19.705 s
PROCESS_FILE:20170906_12_03_24.npz	28.017 s
GPU_COMPUTE	18.157 s
PROCESS_FILE:20170906_12_03_48.npz	29.111 s
GPU_COMPUTE	18.454 s
"""

proc = []
gpu = []

for line in raw_final.strip().splitlines():
    line = line.strip()
    if not line or line.startswith("Name"):
        continue
    # extract the trailing float before ' s'
    m = re.search(r'([0-9]+\.[0-9]+)\s*s$', line)
    if not m:
        continue
    val = float(m.group(1))
    if line.startswith("PROCESS_FILE"):
        proc.append(val)
    elif line.startswith("GPU_COMPUTE"):
        gpu.append(val)

len_proc = len(proc)
len_gpu = len(gpu)

def summarize(vals):
    return {
        "count": len(vals),
        "mean_s": mean(vals),
        "std_sample_s": stdev(vals) if len(vals) > 1 else float('nan'),
        "std_population_s": pstdev(vals) if len(vals) > 1 else float('nan'),
        "min_s": min(vals),
        "max_s": max(vals),
    }

summary = pd.DataFrame.from_dict(
    {"PROCESS_FILE": summarize(proc),
     "GPU_COMPUTE": summarize(gpu)},
    orient="index"
)

# round for readability
summary_rounded = summary.round(3)

print("Timing summary (seconds)")
summary_rounded

Timing summary (seconds)


,count,mean_s,std_sample_s,std_population_s,min_s,max_s
PROCESS_FILE,24,28.993,1.741,1.704,25.717,31.951
GPU_COMPUTE,24,18.694,0.677,0.662,17.528,19.916


In [30]:
# Compute overhead
paired = list(zip(proc, gpu))
if len(paired) == min(len_proc, len_gpu):
    overheads = [p - g for p, g in paired]
    overhead_summary = {
        "count": len(overheads),
        "mean_overhead_s": mean(overheads),
        "std_sample_s": stdev(overheads) if len(overheads) > 1 else float('nan'),
        "min_s": min(overheads),
        "max_s": max(overheads),
    }
    overhead_df = pd.DataFrame([overhead_summary])
    overhead_df = overhead_df.round(3)

print("Non-GPU overhead (PROCESS_FILE - GPU_COMPUTE)")
overhead_df

Non-GPU overhead (PROCESS_FILE - GPU_COMPUTE)


,count,mean_overhead_s,std_sample_s,min_s,max_s
0,24,10.299,1.102,8.058,12.212


In [31]:
# Calculate Corelation Coefficient (COr)
corelation = np.corrcoef(proc, gpu)[0, 1]
print(f"Corelation Coefficient (COr) between PROCESS_FILE and GPU_COMPUTE: {corelation:.3f}")

Corelation Coefficient (COr) between PROCESS_FILE and GPU_COMPUTE: 0.966
